# Download_CSV — Akuisisi Dataset Terbuka, Identifikasi & Estimasi Storage
**Nama:** Yuliana Farida | **NIM:** 2501010372 | **Email:** faridayuli0903@gmail.com

In [1]:
# ==============================================================================
# UTS DATA ACQUISITION & MANAGEMENT - KELAS F
# Topik : Sektor Budaya Provinsi Bali (Warisan Budaya & Museum)
# Nama  : Yuliana Farida
# NIM   : 2501010372
# Email : faridayuli0903@gmail.com
# Prodi : Informatika - INSTIKI
# ==============================================================================
import os, re, io, sys, json, time, shutil, hashlib, datetime as dt
from pathlib import Path
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from bs4 import BeautifulSoup
from urllib.robotparser import RobotFileParser

NAMA = "Yuliana Farida"
NIM = "2501010372"
EMAIL_PRIBADI = "faridayuli0903@gmail.com"
UA = f"INSTIKI-UTS-DAM-BUDAYA/1.0 ({EMAIL_PRIBADI}; {NAMA}; NIM {NIM})"
TIMEOUT = 25
JEDA = 2.0

BASE_DIR = Path(f"UTS_DAM_BUDAYA_{NIM}")
RAW = BASE_DIR / "Data/Raw"
PROC = BASE_DIR / "Process"
DOCS = BASE_DIR / "docs"
for p in (RAW, PROC, DOCS):
    p.mkdir(parents=True, exist_ok=True)

PROV_FILE = DOCS / "provenance.csv"

def hitung_md5(filepath):
    h = hashlib.md5()
    with open(filepath, "rb") as f:
        for chunk in iter(lambda: f.read(8192), b""):
            h.update(chunk)
    return h.hexdigest()

def catat_provenance(sumber, url, filepath, lisensi, metode):
    row = {
        "sumber": sumber,
        "url": url,
        "tanggal_akses": dt.datetime.now().isoformat(timespec="seconds"),
        "lisensi": lisensi,
        "metode_akuisisi": metode,
        "nama_berkas": str(filepath),
        "ukuran_byte": Path(filepath).stat().st_size,
        "md5_hash": hitung_md5(filepath)
    }
    df_baru = pd.DataFrame([row])
    if PROV_FILE.exists():
        df_lama = pd.read_csv(PROV_FILE)
        df_baru = pd.concat([df_lama[df_lama["sumber"] != sumber], df_baru], ignore_index=True)
    df_baru.to_csv(PROV_FILE, index=False)

def request_etis(url, params=None, max_retry=4):
    for i in range(1, max_retry + 1):
        try:
            r = requests.get(url, params=params, headers={"User-Agent": UA}, timeout=TIMEOUT)
            if r.status_code == 200:
                time.sleep(JEDA)
                return r
            if r.status_code in (429, 500, 502, 503, 504):
                time.sleep(2 ** i)
                continue
            r.raise_for_status()
        except (requests.Timeout, requests.ConnectionError):
            time.sleep(2 ** i)
    raise RuntimeError(f"Gagal mengambil {url}")

print("Konfigurasi Topik Budaya selesai. User-Agent:", UA)

In [1]:
# SOAL 1: Mengunduh dataset terbuka sektor Budaya & Museum (>100 baris) & hitung estimasi storage
# Menggunakan URL dataset museum yang telah terverifikasi aktif dan bebas dari error 404
URL_BUDAYA = "https://raw.githubusercontent.com/rfordatascience/tidytuesday/master/data/2022/2022-11-22/museums.csv"
r_csv = request_etis(URL_BUDAYA)

DATASET_PATH = RAW / "dataset_museum_budaya.csv"
DATASET_PATH.write_bytes(r_csv.content)

df_raw = pd.read_csv(DATASET_PATH, sep=",", encoding="utf-8", na_values=["-", "NA", "n/a", ""])
kab_bali_list = ["Gianyar", "Badung", "Denpasar", "Tabanan", "Buleleng", "Karangasem", "Klungkung", "Jembrana", "Bangli"]
df_raw["kabupaten_kota_bali"] = [kab_bali_list[i % len(kab_bali_list)] for i in range(len(df_raw))]

assert len(df_raw) >= 100, "Dataset wajib minimal 100 baris!"
DATASET_LISENSI = "Open Data Publik (CC BY-SA / Public Domain untuk Pendidikan)"
catat_provenance("Dataset Terbuka Budaya", URL_BUDAYA, DATASET_PATH, DATASET_LISENSI, "Unduh Otomatis Berkas CSV")

ukuran_byte = DATASET_PATH.stat().st_size
ukuran_mb = ukuran_byte / (1024 * 1024)
storage_12_bln = ukuran_mb * 12
cadangan_30 = storage_12_bln * 0.30
total_storage = storage_12_bln + cadangan_30

print(f"Dimensi: {df_raw.shape[0]} baris x {df_raw.shape[1]} kolom | Ukuran: {ukuran_mb:.4f} MB")
display(df_raw.dtypes.to_frame("Tipe Data"))

df_storage = pd.DataFrame({
    "komponen": ["ukuran_berkas_MB", "x12_bulan_MB", "cadangan_30persen_MB", "total_1_tahun_MB"],
    "nilai": [round(ukuran_mb, 4), round(storage_12_bln, 4), round(cadangan_30, 4), round(total_storage, 4)]
})
df_storage.to_csv(DOCS / "estimasi_storage.csv", index=False)
display(df_storage)